# Experiment 2 — History-only ablation vs. Frozen Baseline 1 (Qwen2.5-Omni-3B, IEMOCAP)

**Research question:** does adding preceding dialogue context improve zero-shot
Qwen2.5-Omni-3B IEMOCAP emotion recognition, holding the target utterance's audio
and every other variable fixed?

**Design: two matched arms, same session, same loaded model, same 1,622 target ids, same order.**

| | Control — `NoHist` | Treatment — `Hist8` |
|---|---|---|
| Target audio | `audio/<id>.wav` | identical |
| Target transcript | ground-truth (`test.json`) | identical |
| History | none | 8 preceding turns, transcript-only, anonymized `Speaker_1`/`Speaker_2`, current turn excluded |
| Model / weights | `Qwen/Qwen2.5-Omni-3B`, fp16, talker disabled | identical (same loaded instance) |
| Decoding | greedy, `max_new_tokens=12`, seed=11 | identical |
| Parser | **new** 3-tier deterministic parser (`normalize_and_map_answer_v2`) | identical |

**The only intended input difference between the two arms is `history_context`.**
Primary reportable quantity: the **paired** Hist8-vs-NoHist delta under this one
common implementation (`iemocap_eval_lib.paired_comparison`).

**This notebook does NOT recompute, overwrite, or reinterpret Baseline‑1.**
Baseline‑1 (frozen, historical, scored with the *old* parser): WA 38.78%, UA
38.10%, macro‑F1 34.21%, weighted‑F1 33.09%, N=1,622. That run's raw per‑utterance
outputs were never durably persisted, so it cannot be retrospectively re-scored —
**the Control (NoHist) arm here is a *new* run with the *new* parser, not a
substitute for re-scoring Baseline‑1.** Any comparison back to Baseline‑1's
historical numbers must go through this Control arm as an explicit, disclosed
bridge (new parser + fresh sampling of the same model/prompt), not treated as
numerically identical to Baseline‑1.


## 0 · Kaggle setup

1. **Add data:** a dataset containing `test.json`, `test_tiny.json`, `test_history8.json`,
   `test_tiny_history8.json`, `manifest.json`, `manifest_history8.json`, and `audio/`
   (the same 1,622 Session‑5 wavs already used for Baseline 1 — no re-upload needed if
   reusing that dataset version; just add the two new `*history8.json` files to it).
2. **Accelerator:** GPU **T4 x2** (single T4/P100 also works, auto-detected).
3. **Internet:** On.
4. **Secret (optional):** `HF_TOKEN` — Qwen is ungated, not required for this notebook.
5. Run cells top to bottom. Smoke test (36 ids x 2 arms) runs first; review it before the
   full run (1,622 ids x 2 arms, sequential: all Control then all Treatment, same loaded model).

## ⚠️ Output persistence — read this before you start (this is what failed last time)

Baseline‑1's raw per‑utterance outputs were **never recoverable** because they only ever
existed in the live session's `/kaggle/working/` — the session ended without that directory
being captured into a persisted Version, so nothing could be re-scored afterward. This
notebook now has two independent safety nets; **do at least one, ideally both**:

1. **Live, zero-dependency safety net (do this as soon as it appears, don't wait for the
   run to finish):** every full-run cell periodically rewrites a single archive,
   `/kaggle/working/results_bundle.zip`, containing everything produced so far (including
   partial results if the session is interrupted mid-run). In the notebook editor's right
   sidebar, open **Data → Output**, find `results_bundle.zip`, and click **Download** —
   this grabs a local copy immediately, independent of anything below.
2. **Durable, official persistence:** when the full run finishes, use
   **Save Version → "Save & Run All (Commit)"** (top-right of the editor) — **not** "Quick
   Save", which does not reliably persist `/kaggle/working` as that Version's Output. Wait
   for the commit run to finish, then open that Version's page → **Output** tab and confirm
   `results/pred_control_nohist_newparser.jsonl`, `results/pred_treatment_hist8_newparser.jsonl`,
   and `results/paired_comparison.json` are listed there (the notebook's last cell prints an
   explicit checklist and file-size/line-count check for exactly these three files — check
   that it printed no `[MISSING]`/`[EMPTY]` warnings before you close the session).


In [1]:
# ============================ CONFIG ============================
MODEL_ID       = "Qwen/Qwen2.5-Omni-3B"   # fixed -- same weights, both arms
WINDOW         = 8                         # documentation only; already baked into test_history8.json

RUN_SMOKE_TEST = True     # 36-id x 2-arm smoke test first
RUN_FULL       = True     # then 1622-id x 2-arm full run
FORCE_FULL     = False    # True = run full even if the smoke test looked broken

LIMIT          = None     # None = all 1622; or an int for a partial run (applied identically to both arms)
MAX_NEW_TOKENS = 12       # identical to Baseline 1
FORCE_FP16     = None     # None = auto (fp16 on T4, else bf16)
SEED           = 11       # identical to Baseline 1

OUTPUT_DIR     = "/kaggle/working/results"
CONTROL_TAG    = "control_nohist_newparser"
TREATMENT_TAG  = "treatment_hist8_newparser"

EVAL_LIB_SHA256_EXPECTED = "24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298"  # must match the locally-validated file
print("MODEL_ID:", MODEL_ID, "| WINDOW:", WINDOW, "| SEED:", SEED)


MODEL_ID: Qwen/Qwen2.5-Omni-3B | WINDOW: 8 | SEED: 11


In [2]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=4.52.3", "accelerate>=0.34", "qwen-omni-utils", "librosa", "soundfile")
pip("-U", "--force-reinstall", "numpy", "scipy")  # keep numpy/scipy ABI-consistent after librosa pulls a scipy upgrade (fixes: numpy._core._multiarray_umath has no attribute '_blas_supports_fpe')
print("pip done")


pip done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.67.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60

In [3]:
import os, json, time, random, hashlib, base64
import numpy as np
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA        :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("compute cap :", cap)
    _is_t4 = cap[0] == 7
else:
    _is_t4 = False
USE_FP16 = _is_t4 if FORCE_FP16 is None else FORCE_FP16
DTYPE = torch.float16 if USE_FP16 else torch.bfloat16
print("dtype       :", DTYPE)
os.makedirs(OUTPUT_DIR, exist_ok=True)


torch       : 2.10.0+cu128
transformers: 5.17.0
CUDA        : True | GPUs: 2 | Tesla T4
compute cap : (7, 5)
dtype       : torch.float16


In [4]:
# ================= output-persistence safety net =================
# Rewrites ONE archive, /kaggle/working/results_bundle.zip, containing everything under
# OUTPUT_DIR so far. Called periodically during the full run (so an interrupted session
# still leaves a downloadable partial snapshot) and again at the end. Write-to-temp-then-
# rename so an interruption mid-write never leaves a corrupt/half-written zip in its place.
import zipfile

ZIP_PATH = "/kaggle/working/results_bundle.zip"

def make_zip_bundle(note=""):
    tmp = ZIP_PATH + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(OUTPUT_DIR):
            for fn in files:
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, "/kaggle/working"))
    os.replace(tmp, ZIP_PATH)
    sz = os.path.getsize(ZIP_PATH)
    print(f"[checkpoint zip] {ZIP_PATH} ({sz/1e6:.2f} MB){' -- ' + note if note else ''} "
          f"-- download it now from Data > Output if you want an immediate local copy")
    return ZIP_PATH

make_zip_bundle("initial (empty) checkpoint")


[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- initial (empty) checkpoint -- download it now from Data > Output if you want an immediate local copy


'/kaggle/working/results_bundle.zip'

In [5]:
# ---- locate the uploaded dataset (needs BOTH control and treatment files + audio/) ----
CANDIDATE_ROOTS = []
for base in ["/kaggle/input"]:
    if os.path.isdir(base):
        for d in sorted(os.listdir(base)):
            CANDIDATE_ROOTS.append(os.path.join(base, d))

REQUIRED_FILES = ["test.json", "test_history8.json", "test_tiny.json", "test_tiny_history8.json"]

def _find_input_dir():
    for root in CANDIDATE_ROOTS:
        for dirpath, _, files in os.walk(root):
            if all(f in files for f in REQUIRED_FILES) and os.path.isdir(os.path.join(dirpath, "audio")):
                return dirpath
    raise FileNotFoundError(
        f"Could not find all of {REQUIRED_FILES} + audio/ under one directory in /kaggle/input. "
        f"Add a dataset containing the Experiment-2 files (built by kaggle_prep/build_history8_field.py) "
        f"alongside the existing audio/. Seen roots: {CANDIDATE_ROOTS}")

INPUT_DIR = _find_input_dir()
AUDIO_DIR = os.path.join(INPUT_DIR, "audio")
print("INPUT_DIR:", INPUT_DIR)
print("has audio/:", os.path.isdir(AUDIO_DIR), "| n wav:", len(os.listdir(AUDIO_DIR)))


INPUT_DIR: /kaggle/input/datasets/pranavjaiganesh/kaggleupload/kaggle_upload
has audio/: True | n wav: 1622


In [6]:
# ---- HF auth (optional; Qwen is ungated) ----
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)


HF_TOKEN loaded from Kaggle Secrets.


In [7]:
# ================= iemocap_eval_lib.py, embedded verbatim as base64 =================
# This is the EXACT byte content of kaggle_prep/iemocap_eval_lib.py as locally validated
# (unit-tested + file-level assertions run) in this session. Base64 (not a triple-quoted
# string) is used because the file's own docstrings contain triple-quotes, which would
# break naive string embedding. sha256 is checked below so any transcription drift is
# caught loudly rather than silently.
EVAL_LIB_B64 = "..."

_lib_bytes = base64.b64decode(EVAL_LIB_B64)
_lib_sha256 = hashlib.sha256(_lib_bytes).hexdigest()
print("iemocap_eval_lib.py decoded:", len(_lib_bytes), "bytes, sha256=", _lib_sha256)
assert _lib_sha256 == EVAL_LIB_SHA256_EXPECTED, (
    "iemocap_eval_lib.py content does not match the locally-validated version! "
    f"expected {EVAL_LIB_SHA256_EXPECTED}, got {_lib_sha256}")
print("sha256 matches the locally-validated file -- OK")

with open("/kaggle/working/iemocap_eval_lib.py", "wb") as f:
    f.write(_lib_bytes)
import importlib, sys
sys.path.insert(0, "/kaggle/working")
import iemocap_eval_lib as EVAL
importlib.reload(EVAL)
print("eval lib ready. labels:", EVAL.IEMOCAP_LABELS)


iemocap_eval_lib.py decoded: 24510 bytes, sha256= 24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298
sha256 matches the locally-validated file -- OK
eval lib ready. labels: ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']


In [8]:
# ---- load records for BOTH arms, and ASSERT the pairing invariants at runtime ----
def load_records(name):
    with open(os.path.join(INPUT_DIR, name), encoding="utf-8") as f:
        return json.load(f)

CONTROL_TINY   = load_records("test_tiny.json")
TREATMENT_TINY = load_records("test_tiny_history8.json")
CONTROL_FULL   = load_records("test.json")
TREATMENT_FULL = load_records("test_history8.json")
if LIMIT:
    CONTROL_FULL = CONTROL_FULL[:LIMIT]
    TREATMENT_FULL = TREATMENT_FULL[:LIMIT]

def assert_paired(control, treatment, tag):
    assert len(control) == len(treatment), f"{tag}: length mismatch {len(control)} vs {len(treatment)}"
    cids = [r["id"] for r in control]; tids = [r["id"] for r in treatment]
    assert cids == tids, f"{tag}: id order mismatch"
    for c, t in zip(control, treatment):
        assert c["utterance"] == t["utterance"], f"{tag}: utterance diverged for {c['id']}"
        assert c["output"] == t["output"], f"{tag}: gold label diverged for {c['id']}"
        assert set(t.keys()) - set(c.keys()) == {"history_context"}, \
            f"{tag}: unexpected extra fields for {c['id']}: {set(t.keys())-set(c.keys())}"
    return cids

tiny_ids = assert_paired(CONTROL_TINY, TREATMENT_TINY, "tiny")
full_ids = assert_paired(CONTROL_FULL, TREATMENT_FULL, "full")
print(f"ASSERTIONS PASSED: tiny {len(tiny_ids)} ids paired & identical (except history_context); "
      f"full {len(full_ids)} ids paired & identical (except history_context)")
n_empty_hist = sum(1 for r in TREATMENT_FULL if r["history_context"] == "")
print(f"true dialogue-openers in full set (empty history_context, both arms structurally identical for these): {n_empty_hist}")


ASSERTIONS PASSED: tiny 36 ids paired & identical (except history_context); full 1622 ids paired & identical (except history_context)
true dialogue-openers in full set (empty history_context, both arms structurally identical for these): 22


In [9]:
# ================= model loading (ONCE, shared by both arms) =================
_MODEL = {}

def load_model():
    from transformers import Qwen2_5OmniProcessor
    from qwen_omni_utils import process_mm_info
    import transformers as _tf
    OmniCls = getattr(_tf, "Qwen2_5OmniForConditionalGeneration",
                      getattr(_tf, "Qwen2_5OmniModel", None))
    if OmniCls is None:
        raise ImportError("This transformers build has no Qwen2.5-Omni class.")
    proc = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
    # NOTE: not passing enable_audio_output=False / calling disable_talker(): this
    # transformers build's generate() unconditionally reads self.talker.codec_pad_token
    # before checking return_audio, so a talker-less model crashes generate() even with
    # return_audio=False (already passed below). Loading it fully does not change the
    # generated text, since only self.thinker.generate() produces the returned tokens.
    model = OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
    model.eval()
    _MODEL.update(model=model, proc=proc, process_mm_info=process_mm_info)
    print("model loaded:", MODEL_ID, "(one instance, shared by both arms)")

load_model()


[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded: Qwen/Qwen2.5-Omni-3B (one instance, shared by both arms)


In [10]:
# ================= per-utterance inference (history_context passed explicitly) =================
QWEN_SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
            "capable of perceiving auditory and visual inputs, as well as generating text and speech.")

@torch.no_grad()
def infer_one(rec, history_context):
    """history_context: None/""/str. Passed explicitly per call so the SAME model
    instance can serve both arms in one session without any global mode flag."""
    prompt = EVAL.build_prompt(rec["utterance"], history_context)
    wav_path = os.path.join(AUDIO_DIR, rec["id"] + ".wav")
    model, proc, process_mm_info = _MODEL["model"], _MODEL["proc"], _MODEL["process_mm_info"]

    conv = [
        {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": wav_path},
            {"type": "text", "text": prompt}]},
    ]
    text = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    try:
        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    except TypeError:
        audios, images, videos = process_mm_info(conv)
    try:
        inputs = proc(text=text, audio=audios, images=images, videos=videos,
                      return_tensors="pt", padding=True)
    except TypeError:
        inputs = proc(text=text, audios=audios, images=images, videos=videos,
                      return_tensors="pt", padding=True)
    inputs = inputs.to(model.device)
    try:
        inputs = inputs.to(model.dtype)
    except Exception:
        pass
    gkw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    try:
        out = model.generate(**inputs, return_audio=False, **gkw)
    except TypeError:
        out = model.generate(**inputs, **gkw)
    if isinstance(out, (tuple, list)):
        out = out[0]
    gen = out[:, inputs["input_ids"].shape[1]:]
    return proc.batch_decode(gen, skip_special_tokens=True,
                             clean_up_tokenization_spaces=False)[0].strip()

# one-shot sanity check per arm before the smoke test
t0 = time.time()
_demo_c = infer_one(CONTROL_TINY[0], None)
_demo_t = infer_one(TREATMENT_TINY[0], TREATMENT_TINY[0]["history_context"])
print(f"1st Control   inference OK in {time.time()-t0:.1f}s -> id={CONTROL_TINY[0]['id']} gold={CONTROL_TINY[0]['output']} raw={_demo_c!r}")
t0 = time.time()
print(f"1st Treatment inference OK in {time.time()-t0:.1f}s -> id={TREATMENT_TINY[0]['id']} gold={TREATMENT_TINY[0]['output']} raw={_demo_t!r}")


1st Control   inference OK in 5.5s -> id=Ses05F_impro01_F000 gold=neutral raw='Neutral. What do you think about this?'
1st Treatment inference OK in 0.0s -> id=Ses05F_impro01_F000 gold=neutral raw='Neutral. What do you think about this?'


In [11]:
# ================= SMOKE TEST: both arms, 36 ids =================
from tqdm.auto import tqdm

def run_set(control_recs, treatment_recs, desc):
    c_raw, t_raw = [], []
    for r in tqdm(control_recs, desc=f"{desc}-control"):
        try: c_raw.append(infer_one(r, None))
        except Exception as e: print("control infer error", r["id"], repr(e)); c_raw.append("")
    for r in tqdm(treatment_recs, desc=f"{desc}-treatment"):
        try: t_raw.append(infer_one(r, r["history_context"]))
        except Exception as e: print("treatment infer error", r["id"], repr(e)); t_raw.append("")
    return c_raw, t_raw

SMOKE_OK = None
if RUN_SMOKE_TEST:
    c_raw, t_raw = run_set(CONTROL_TINY, TREATMENT_TINY, "smoke")
    c_m = EVAL.score_predictions_normalized(CONTROL_TINY, c_raw)
    t_m = EVAL.score_predictions_normalized(TREATMENT_TINY, t_raw)
    print(f"{'id':<24}{'gold':<12}{'ctrl_pred':<12}{'ctrl_tier':<22}{'trt_pred':<12}{'trt_tier':<22}")
    for i, uid in enumerate(tiny_ids):
        cp = c_m["predictions"][i]; tp = t_m["predictions"][i]
        print(f"{uid:<24}{cp['gold']:<12}{cp['normalized_prediction']:<12}{cp['match_tier']:<22}"
              f"{tp['normalized_prediction']:<12}{tp['match_tier']:<22}")
    n_empty_c = sum(1 for x in c_raw if not x.strip())
    n_empty_t = sum(1 for x in t_raw if not x.strip())
    SMOKE_OK = (n_empty_c <= 2) and (n_empty_t <= 2)
    print(f"\ncontrol:   WA={c_m['accuracy_WA']:.2f} macro-F1={c_m['macro_f1']:.2f} tiers={c_m['match_tier_counts']}")
    print(f"treatment: WA={t_m['accuracy_WA']:.2f} macro-F1={t_m['macro_f1']:.2f} tiers={t_m['match_tier_counts']}")
    print(f"\nSMOKE_OK = {SMOKE_OK}")
    with open(os.path.join(OUTPUT_DIR, "smoke_control.jsonl"), "w", encoding="utf-8") as f:
        for uid, raw in zip(tiny_ids, c_raw): f.write(json.dumps({"id": uid, "raw": raw}, ensure_ascii=False)+"\n")
    with open(os.path.join(OUTPUT_DIR, "smoke_treatment.jsonl"), "w", encoding="utf-8") as f:
        for uid, raw in zip(tiny_ids, t_raw): f.write(json.dumps({"id": uid, "raw": raw}, ensure_ascii=False)+"\n")
    make_zip_bundle("after smoke test -- also exercises the zip mechanism before the long run")
else:
    print("smoke test skipped")


smoke-control:   0%|          | 0/36 [00:00<?, ?it/s]

smoke-treatment:   0%|          | 0/36 [00:00<?, ?it/s]

id                      gold        ctrl_pred   ctrl_tier             trt_pred    trt_tier              
Ses05F_impro01_F000     neutral     neutral     word_match            neutral     word_match            
Ses05F_impro01_M000     neutral     frustrated  word_match            neutral     exact                 
Ses05F_impro01_F001     frustrated  frustrated  word_match            frustrated  word_match            
Ses05F_impro01_F002     frustrated  frustrated  word_match            frustrated  word_match            
Ses05F_impro01_F003     frustrated  frustrated  word_match            frustrated  exact                 
Ses05F_impro01_M003     neutral     frustrated  word_match            frustrated  exact                 
Ses05F_impro01_F004     frustrated  frustrated  word_match            frustrated  word_match            
Ses05F_impro01_M004     frustrated  neutral     word_match            neutral     word_match            
Ses05F_impro01_F005     frustrated  frustrated  word_ma

In [12]:
# ================= FULL RUN: both arms, sequential (control then treatment), same model =================
if RUN_FULL and RUN_SMOKE_TEST and (SMOKE_OK is False) and not FORCE_FULL:
    raise RuntimeError("Smoke test looked broken. Not starting the full run. Set FORCE_FULL=True to override.")

CONTROL_JSONL   = os.path.join(OUTPUT_DIR, f"pred_{CONTROL_TAG}.jsonl")
TREATMENT_JSONL = os.path.join(OUTPUT_DIR, f"pred_{TREATMENT_TAG}.jsonl")

def load_done(path):
    done = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    o = json.loads(line); done[o["id"]] = o["raw"]
    return done

ZIP_EVERY = 200  # rows between checkpoint zips during the full run (jsonl itself is flushed every row)

def run_full_arm(records, jsonl_path, get_history, desc):
    done = load_done(jsonl_path)
    print(f"[{desc}] resuming: {len(done)} / {len(records)} already done")
    t0 = time.time()
    with open(jsonl_path, "a", encoding="utf-8") as f:
        for i, r in enumerate(tqdm(records, desc=desc)):
            if r["id"] in done:
                continue
            try:
                raw = infer_one(r, get_history(r))
            except Exception as e:
                print(f"[{desc}] infer error", r["id"], repr(e)); raw = ""
            done[r["id"]] = raw
            f.write(json.dumps({"id": r["id"], "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if (i + 1) % 100 == 0:
                el = time.time() - t0
                print(f"  [{desc}] {i+1}/{len(records)}  elapsed {el/60:.1f}m  eta {el/(i+1)*(len(records)-i-1)/60:.1f}m")
            if (i + 1) % ZIP_EVERY == 0:
                make_zip_bundle(f"mid-run checkpoint, {desc} {i+1}/{len(records)}")
    print(f"[{desc}] done in {(time.time()-t0)/60:.1f} min -> {jsonl_path}")
    make_zip_bundle(f"{desc} arm complete")
    return done

if RUN_FULL:
    control_done = run_full_arm(CONTROL_FULL, CONTROL_JSONL, lambda r: None, "control(NoHist)")
    # checkpoint explicitly named here too -- this is the file to grab if the session dies
    # during the (longer, second) treatment run: the control arm's raw predictions are safe.
    make_zip_bundle("control arm complete, about to start treatment arm")
    treatment_done = run_full_arm(TREATMENT_FULL, TREATMENT_JSONL, lambda r: r["history_context"], "treatment(Hist8)")
else:
    print("full run skipped")


[control(NoHist)] resuming: 0 / 1622 already done


control(NoHist):   0%|          | 0/1622 [00:00<?, ?it/s]

  [control(NoHist)] 100/1622  elapsed 2.6m  eta 39.4m
  [control(NoHist)] 200/1622  elapsed 4.7m  eta 33.3m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- mid-run checkpoint, control(NoHist) 200/1622 -- download it now from Data > Output if you want an immediate local copy
  [control(NoHist)] 300/1622  elapsed 7.0m  eta 31.0m
  [control(NoHist)] 400/1622  elapsed 9.4m  eta 28.6m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- mid-run checkpoint, control(NoHist) 400/1622 -- download it now from Data > Output if you want an immediate local copy
  [control(NoHist)] 500/1622  elapsed 11.4m  eta 25.6m
  [control(NoHist)] 600/1622  elapsed 13.7m  eta 23.3m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- mid-run checkpoint, control(NoHist) 600/1622 -- download it now from Data > Output if you want an immediate local copy
  [control(NoHist)] 700/1622  elapsed 15.6m  eta 20.6m
  [control(NoHist)] 800/1622  elapsed 17.7m  eta 18.2m
[checkpoint 

treatment(Hist8):   0%|          | 0/1622 [00:00<?, ?it/s]

  [treatment(Hist8)] 100/1622  elapsed 2.4m  eta 37.0m
  [treatment(Hist8)] 200/1622  elapsed 4.1m  eta 29.0m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.01 MB) -- mid-run checkpoint, treatment(Hist8) 200/1622 -- download it now from Data > Output if you want an immediate local copy
  [treatment(Hist8)] 300/1622  elapsed 6.1m  eta 26.7m
  [treatment(Hist8)] 400/1622  elapsed 8.2m  eta 25.1m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.01 MB) -- mid-run checkpoint, treatment(Hist8) 400/1622 -- download it now from Data > Output if you want an immediate local copy
  [treatment(Hist8)] 500/1622  elapsed 9.9m  eta 22.3m
  [treatment(Hist8)] 600/1622  elapsed 12.0m  eta 20.4m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.01 MB) -- mid-run checkpoint, treatment(Hist8) 600/1622 -- download it now from Data > Output if you want an immediate local copy
  [treatment(Hist8)] 700/1622  elapsed 13.8m  eta 18.2m
  [treatment(Hist8)] 800/1622  elapsed 15.6m  eta 16.0m
[c

In [13]:
# ================= SCORE + SAVE both arms + PRIMARY paired comparison =================
import pandas as pd

def read_jsonl(path):
    d = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                o = json.loads(line); d[o["id"]] = o["raw"]
    return d

control_done = read_jsonl(CONTROL_JSONL)
treatment_done = read_jsonl(TREATMENT_JSONL)

c_missing = [r["id"] for r in CONTROL_FULL if r["id"] not in control_done]
t_missing = [r["id"] for r in TREATMENT_FULL if r["id"] not in treatment_done]
if c_missing or t_missing:
    print(f"WARNING: control missing {len(c_missing)}, treatment missing {len(t_missing)} -- "
          f"re-run the full-run cell to finish before trusting the paired comparison.")

c_scored = [r for r in CONTROL_FULL if r["id"] in control_done]
t_scored = [r for r in TREATMENT_FULL if r["id"] in treatment_done]
c_raws = [control_done[r["id"]] for r in c_scored]
t_raws = [treatment_done[r["id"]] for r in t_scored]

C = EVAL.score_predictions_normalized(c_scored, c_raws)
T = EVAL.score_predictions_normalized(t_scored, t_raws)

def banner(tag, M):
    lines = [f"=== {tag} (N={M['n_samples']}) ==="]
    lines.append(f"Accuracy/WA={M['accuracy_WA']:.2f}  UA={M['UA_unweighted_recall']:.2f}  "
                 f"macro-F1={M['macro_f1']:.2f}  weighted-F1={M['weighted_f1']:.2f}")
    lines.append(f"match tiers: {M['match_tier_counts']}  ambiguous_multi_label={M['n_ambiguous_multi_label']}")
    lines.append(f"repo report_score: {M['repo_report_score']}")
    return "\n".join(lines)

print(banner("CONTROL (NoHist, new parser)", C))
print()
print(banner("TREATMENT (Hist8, new parser)", T))

print("\n" + "="*78)
print("PRIMARY RESULT: paired Treatment(Hist8) - Control(NoHist) delta, same model/ids/parser")
print("="*78)
if not c_missing and not t_missing:
    P = EVAL.paired_comparison([r["id"] for r in CONTROL_FULL], C, [r["id"] for r in TREATMENT_FULL], T)
    print(json.dumps({k: v for k, v in P.items() if k != "flips"}, indent=2))
    print(f"\nflips: {len(P['flips'])} targets changed correctness between arms "
          f"(only_control_correct={P['only_control_correct']}, only_treatment_correct={P['only_treatment_correct']})")
else:
    P = None
    print("SKIPPED -- control/treatment have missing rows, arms not yet fully paired. "
          "Finish the full run for both arms before computing the paired comparison.")

print("\n" + "="*78)
print("REMINDER: Baseline-1 (historical, old parser) is NOT recomputed or overwritten here.")
print("Baseline-1: WA 38.78%, UA 38.10%, macro-F1 34.21%, weighted-F1 33.09%, N=1622 (old parser).")
print("Any comparison to Baseline-1 must go through CONTROL above as an explicit bridge,")
print("acknowledging the parser difference -- it is not a numerically clean re-score of Baseline-1.")
print("="*78)

for tag, M in [(CONTROL_TAG, C), (TREATMENT_TAG, T)]:
    with open(os.path.join(OUTPUT_DIR, f"metrics_{tag}.json"), "w", encoding="utf-8") as f:
        json.dump({k: v for k, v in M.items() if k != "predictions"}, f, ensure_ascii=False, indent=2)
    with open(os.path.join(OUTPUT_DIR, f"predictions_{tag}.json"), "w", encoding="utf-8") as f:
        json.dump(M["predictions"], f, ensure_ascii=False, indent=2)
    pd.DataFrame(M["confusion_matrix"]["rows_gold_cols_pred"],
                 index=EVAL.IEMOCAP_LABELS, columns=EVAL.IEMOCAP_LABELS
                 ).to_csv(os.path.join(OUTPUT_DIR, f"confusion_{tag}.csv"))
if P is not None:
    with open(os.path.join(OUTPUT_DIR, "paired_comparison.json"), "w", encoding="utf-8") as f:
        json.dump(P, f, ensure_ascii=False, indent=2)
run_meta = {"model_id": MODEL_ID, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
            "window": WINDOW, "dtype": str(DTYPE), "transformers": transformers.__version__,
            "torch": torch.__version__, "eval_lib_sha256": _lib_sha256,
            "n_control": len(c_scored), "n_treatment": len(t_scored),
            "control_missing": c_missing, "treatment_missing": t_missing}
with open(os.path.join(OUTPUT_DIR, "run_meta_experiment2.json"), "w", encoding="utf-8") as f:
    json.dump(run_meta, f, ensure_ascii=False, indent=2)
print("\nsaved to", OUTPUT_DIR, "->", sorted(os.listdir(OUTPUT_DIR)))
make_zip_bundle("final, after scoring both arms")


=== CONTROL (NoHist, new parser) (N=1622) ===
Accuracy/WA=41.68  UA=40.13  macro-F1=38.28  weighted-F1=38.76
match tiers: {'exact': 352, 'word_match': 1267, 'edit_distance_fallback': 3}  ambiguous_multi_label=1
repo report_score: {'Acc_SA': 41.677, 'F1_SA': 38.761, 'mode': 'test'}

=== TREATMENT (Hist8, new parser) (N=1622) ===
Accuracy/WA=44.88  UA=44.78  macro-F1=42.16  weighted-F1=41.85
match tiers: {'exact': 695, 'word_match': 924, 'edit_distance_fallback': 3}  ambiguous_multi_label=0
repo report_score: {'Acc_SA': 44.883, 'F1_SA': 41.848, 'mode': 'test'}

PRIMARY RESULT: paired Treatment(Hist8) - Control(NoHist) delta, same model/ids/parser
{
  "n": 1622,
  "delta_accuracy_WA": 3.206,
  "delta_UA": 4.647,
  "delta_macro_f1": 3.883,
  "delta_weighted_f1": 3.087,
  "both_correct": 591,
  "both_wrong": 809,
  "only_control_correct": 85,
  "only_treatment_correct": 137
}

flips: 222 targets changed correctness between arms (only_control_correct=85, only_treatment_correct=137)

REMINDER

'/kaggle/working/results_bundle.zip'

In [14]:
# ================= FINAL PERSISTENCE CHECK -- do not skip this cell =================
# Verifies the files you explicitly said matter most, refreshes the final zip, and prints
# an unmissable checklist. Non-fatal (prints CRITICAL warnings rather than raising) --
# raising here would not recover anything; the point is to make any gap visible NOW,
# while the session is still alive and something can still be done about it.
def _line_count(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

CRITICAL_FILES = {
    "Control raw predictions (jsonl)":   CONTROL_JSONL,
    "Treatment raw predictions (jsonl)": TREATMENT_JSONL,
    "Paired comparison (primary result)": os.path.join(OUTPUT_DIR, "paired_comparison.json"),
}
EXPECTED_ROWS = len(CONTROL_FULL)  # == len(TREATMENT_FULL), asserted earlier; respects LIMIT

print("="*78)
print("FINAL PERSISTENCE CHECK")
print("="*78)
all_ok = True
for label, path in CRITICAL_FILES.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    if not exists:
        print(f"[MISSING] {label}: {path} does not exist")
        all_ok = False
    elif size == 0:
        print(f"[EMPTY]   {label}: {path} exists but is 0 bytes")
        all_ok = False
    else:
        if path.endswith(".jsonl"):
            n = _line_count(path)
            note = f"{n}/{EXPECTED_ROWS} rows" + ("" if n == EXPECTED_ROWS else "  <-- INCOMPLETE, re-run the full-run cell")
            if n != EXPECTED_ROWS:
                all_ok = False
            print(f"[ok]      {label}: {size/1e6:.2f} MB, {note}")
        else:
            print(f"[ok]      {label}: {size/1e3:.1f} KB")

zpath = make_zip_bundle("FINAL")
print(f"\nfull results/ listing: {sorted(os.listdir(OUTPUT_DIR))}")

print("\n" + "="*78)
if all_ok:
    print("ALL CRITICAL FILES PRESENT AND COMPLETE.")
else:
    print("CRITICAL: one or more files above are MISSING/EMPTY/INCOMPLETE.")
    print("Do not close this session yet -- re-run the full-run cell to finish, then re-run this cell.")
print("="*78)
print("""
NOW DO ONE OR BOTH OF THE FOLLOWING BEFORE CLOSING THIS SESSION:

  1) IMMEDIATE: sidebar -> Data -> Output -> find "results_bundle.zip" -> Download.
     This is a live local copy, independent of everything below.

  2) DURABLE: click "Save Version" (top right) -> choose "Save & Run All (Commit)"
     (NOT "Quick Save"). Wait for it to finish, then open that Version's page ->
     Output tab, and confirm you see:
       results/pred_control_nohist_newparser.jsonl
       results/pred_treatment_hist8_newparser.jsonl
       results/paired_comparison.json
       results_bundle.zip
""")


FINAL PERSISTENCE CHECK
[ok]      Control raw predictions (jsonl): 0.16 MB, 1622/1622 rows
[ok]      Treatment raw predictions (jsonl): 0.15 MB, 1622/1622 rows
[ok]      Paired comparison (primary result): 20.7 KB
[checkpoint zip] /kaggle/working/results_bundle.zip (0.05 MB) -- FINAL -- download it now from Data > Output if you want an immediate local copy

full results/ listing: ['confusion_control_nohist_newparser.csv', 'confusion_treatment_hist8_newparser.csv', 'metrics_control_nohist_newparser.json', 'metrics_treatment_hist8_newparser.json', 'paired_comparison.json', 'pred_control_nohist_newparser.jsonl', 'pred_treatment_hist8_newparser.jsonl', 'predictions_control_nohist_newparser.json', 'predictions_treatment_hist8_newparser.json', 'run_meta_experiment2.json', 'smoke_control.jsonl', 'smoke_treatment.jsonl']

ALL CRITICAL FILES PRESENT AND COMPLETE.

NOW DO ONE OR BOTH OF THE FOLLOWING BEFORE CLOSING THIS SESSION:

  1) IMMEDIATE: sidebar -> Data -> Output -> find "results_bundle.

## Interpretation notes

- The **paired comparison** (`paired_comparison.json`) is the primary result: it holds
  weights, target audio, target transcript, ordering, decoding, label mapping and parser
  fixed across arms, varying only `history_context`. `only_treatment_correct` /
  `only_control_correct` are the McNemar b/c cells if a significance test is wanted later.
- **22/1,622 (1.4%) targets are true dialogue-openers** (no preceding turns exist) — for
  those, Control and Treatment prompts are provably identical (`build_prompt` routes an
  empty `history_context` to the no-history branch). This slightly dilutes any aggregate
  Hist8-vs-NoHist effect and should be kept in mind when interpreting the paired delta.
- Do **not** read the Control arm's numbers here as "Baseline‑1, corrected." It is a new
  run, same prompt/model/protocol as Baseline‑1 but scored with the new parser — useful as
  a same-parser reference point for the Treatment arm, not as a retroactive fix to the
  historical Baseline‑1 numbers (which remain frozen and unscored by this notebook).
